# Chapter 6 — Triple-mediated retrieval

This is the thesis chapter. A question is not handed to a vector index; it
is **translated into query triples** — `(head, relation, tail)` patterns in
which exactly the asked-for value is the bare variable `?`. Those triples are
then bound to the graph's vocabulary and answered *through* the GMS. The
natural-language surface form never touches an opaque similarity search.

Ground truth for every listing here is `data/corpus_facts.md` (Northwind
Industries FY2025). We do **not** load the trained store or run Qwen in the
deterministic cells — those are tagged `ci-gpu` for the lead to execute.

In [ ]:
# knowlytix and forgeloop are installed from PyPI (pip install knowlytix forgeloop)
import os

## The query-triple vocabulary
Three symbols carry the chapter: `QueryTriple` (the pattern), `ASKED` (the
bare `?` sentinel), and `QueryTripleExtractor` (NL → triples, LLM injectable).
`schema_from_store` reads the graph's real relation/entity names so the
extractor is *grounded* — the single biggest reliability lever for a small
model (it otherwise invents relation names that never bind).

In [ ]:
from knowlytix.knowledge.rag.query_triples import (
    ASKED, QueryTripleExtractor, is_var, schema_from_store,
)
from knowlytix.knowledge.llm_backend import LLMBackend

print("asked-slot sentinel:", repr(ASKED))
print("is_var('?'):", is_var(ASKED), "| is_var('?x'):", is_var("?x"),
      "| is_var('cloud platform'):", is_var("cloud platform"))

## A deterministic LLM for CI
`QueryTripleExtractor` takes any `LLMBackend`. For reproducible CI we inject
a scripted backend that replays the JSON a correctly-grounded Qwen returns
for our five questions. The **real Qwen path** is shown further down
(`LocalTransformersBackend`, tagged `ci-gpu`). The extractor code is identical;
only the backend swaps — that is the whole point of the injection seam.

In [ ]:
class ScriptedBackend(LLMBackend):
    """Replays canned JSON keyed by question — stands in for Qwen in CI."""

    def __init__(self, script: dict[str, str], default: str = "[]"):
        self._script = script
        self._default = default

    def call(self, system: str, user: str, max_tokens: int = 2048) -> str:
        question = user.split("\n\nNote:")[0].strip()
        return self._script.get(question, self._default)

    @property
    def model_name(self) -> str:
        return "scripted/qwen2.5-3b-instruct"


# Canned extractions: the JSON a schema-grounded Qwen returns for each question.
SCRIPT = {
    "What was Cloud Platform revenue?":
        '[{"head":"cloud platform","relation":"has_revenue","tail":"?"}]',
    "What was total revenue for FY2025?":
        '[{"head":"total","relation":"has_revenue","tail":"?"}]',
    "How many people work in Logistics?":
        '[{"head":"logistics","relation":"has_headcount","tail":"?"}]',
    "Which division does Cloud Platform belong to?":
        '[{"head":"cloud platform","relation":"has_division","tail":"?"}]',
    "In which region is Cloud Platform's division?":
        '[{"head":"cloud platform","relation":"has_division","tail":"?x"},'
        '{"head":"?x","relation":"has_region","tail":"?"}]',
}
extractor = QueryTripleExtractor(ScriptedBackend(SCRIPT))

## Listing 1 — five questions become query triples
Factual, numeric and multi-hop questions all reduce to triple patterns. In
every case the asked value is the bare `?`; the multi-hop question links two
triples through the named variable `?x`. We print each triple and assert that
every extraction exposes exactly one `?` asked slot.

In [ ]:
QUESTIONS = [
    "What was Cloud Platform revenue?",          # factual
    "What was total revenue for FY2025?",        # numeric / exact
    "How many people work in Logistics?",        # factual count
    "Which division does Cloud Platform belong to?",  # hierarchy
    "In which region is Cloud Platform's division?",  # multi-hop (?x -> ?)
]

for q in QUESTIONS:
    qts = extractor.extract(q)
    asked = sum(t.tail == ASKED for t in qts)
    print(f"Q: {q}")
    for t in qts:
        print(f"   {t.as_tuple()}")
    print(f"   asked-slots: {asked}, intermediate vars: "
          f"{sorted({s for t in qts for s in t.as_tuple() if is_var(s) and s != ASKED})}\n")
    assert asked == 1, f"expected exactly one '?' slot for: {q}"

The multi-hop question is the one to study. `In which region is Cloud
Platform's division?` cannot be a single `(head, relation, ?)` lookup: the
graph has no `cloud platform -> has_region` edge. Instead the extractor emits
a **two-triple chain** — `cloud platform has_division ?x`, then
`?x has_region ?` — where `?x` is the division to be resolved in the first
hop and substituted into the second. Per `corpus_facts.md`, `?x` binds to
`technology` and the final `?` resolves to `north america`.

In [ ]:
multi = extractor.extract("In which region is Cloud Platform's division?")
assert len(multi) == 2
assert multi[0].tail == "?x" and multi[1].head == "?x"   # the chain link
assert multi[1].tail == ASKED                            # the asked value
print("hop 1:", multi[0].as_tuple())
print("hop 2:", multi[1].as_tuple())
print("expected resolution (corpus_facts): ?x=technology, ?=north america")

## Listing 2 — schema grounding fixes a non-binding relation
Left to its own vocabulary, a small model emits plausible-but-wrong relation
names: `headcount`, `staff_count`, `topline`. None of those exist in the
graph (the real relations are `has_revenue`, `has_headcount`, `has_division`,
`has_region`, ...). `schema_from_store` injects the actual relation names into
the prompt so the model emits binding-compatible triples.

We simulate both worlds with two scripted backends — *ungrounded* (the model
guesses `staff_count`) and *grounded* (it picks the real `has_headcount`) —
to show the failure mode and its fix without burning the GPU. The real
grounded Qwen run is the `ci-gpu` cell below.

In [ ]:
# Same question, two model behaviours.
UNGROUNDED = {"How many people work in Logistics?":
              '[{"head":"logistics","relation":"staff_count","tail":"?"}]'}
GROUNDED = {"How many people work in Logistics?":
            '[{"head":"logistics","relation":"has_headcount","tail":"?"}]'}

# The graph's real relation vocabulary (from corpus_facts.md / schema_from_store).
GRAPH_RELATIONS = ["has_amount", "has_division", "has_fy2024", "has_fy2025",
                   "has_head", "has_headcount", "has_region", "has_revenue",
                   "has_value"]

def relation_binds(rel: str) -> bool:
    return rel in GRAPH_RELATIONS

q = "How many people work in Logistics?"
bad = QueryTripleExtractor(ScriptedBackend(UNGROUNDED)).extract(q)[0]
good = QueryTripleExtractor(ScriptedBackend(GROUNDED)).extract(q)[0]

print("ungrounded relation:", bad.relation, "-> binds?", relation_binds(bad.relation))
print("grounded   relation:", good.relation, "-> binds?", relation_binds(good.relation))
assert not relation_binds(bad.relation)   # staff_count is not in the graph
assert relation_binds(good.relation)      # has_headcount is

`schema_from_store(store)` is what produces that relation list at run time.
It returns `{'relations': [...], 'entities': [...]}`, filtering out the
`in_section` bookkeeping relation and any numeric value-entities (those are
answers, not query terms). Passed as `vocab=`, the extractor folds it into the
system prompt. We show the expected shape against `corpus_facts.md` without
loading the store.

In [ ]:
expected_vocab = {
    "relations": ["has_amount", "has_division", "has_fy2024", "has_fy2025",
                  "has_head", "has_headcount", "has_region", "has_revenue",
                  "has_value"],
    "entities": ["cloud platform", "devices", "logistics", "retail",
                 "technology", "operations", "total", "revenue",
                 "net income", "operating income"],
}
grounded_extractor = QueryTripleExtractor(
    ScriptedBackend(SCRIPT), vocab=expected_vocab
)
# The vocab block is injected into the system prompt the extractor sends.
sys_prompt = grounded_extractor.llm  # backend; prompt assembled inside .extract
assert "has_headcount" in expected_vocab["relations"]
assert "in_section" not in expected_vocab["relations"]   # filtered out
print("grounded relations:", expected_vocab["relations"])

## The query-only repair fallback
Grounding reduces but does not eliminate mis-extraction. `RagPipeline`'s
`_extract_and_bind` runs a **query-only repair loop**: if a non-variable slot
fails to bind, it re-asks the extractor once with a hint listing the terms
that did not match and the real relations to use instead. This is a
*query-side* repair — it never edits the graph (that is GEODE's job, Ch. 5).
We illustrate the two-pass behaviour with a backend that returns a bad
relation first and the corrected one when a `Note:` hint is present.

In [ ]:
class RepairingBackend(LLMBackend):
    """First call mis-extracts; a follow-up with a hint corrects it."""

    def call(self, system: str, user: str, max_tokens: int = 2048) -> str:
        if "Note:" in user:   # the repair retry carries a hint
            return '[{"head":"logistics","relation":"has_headcount","tail":"?"}]'
        return '[{"head":"logistics","relation":"headcount","tail":"?"}]'

    @property
    def model_name(self) -> str:
        return "scripted/repairing"


rb = QueryTripleExtractor(RepairingBackend())
first = rb.extract("How many people work in Logistics?")[0]
repaired = rb.extract("How many people work in Logistics?",
                      hint="These terms did not match: ['headcount']. "
                           "Re-express using ONLY: has_headcount.")[0]
print("first pass :", first.as_tuple(), "-> binds?", relation_binds(first.relation))
print("after repair:", repaired.as_tuple(), "-> binds?", relation_binds(repaired.relation))
assert not relation_binds(first.relation)
assert relation_binds(repaired.relation)

## The real Qwen path (CI-only)
Everything above used scripted backends for determinism. Here is the
production wiring: a local **Qwen3-4B-Instruct** via `LocalTransformersBackend`,
grounded with the live `schema_from_store`. This cell loads the trained store
and a GPU model, so it is tagged `ci-gpu` — the lead executes it; do not run
it in a shared-GPU authoring session.

In [ ]:
# CI-ONLY (ci-gpu): loads the trained store + a GPU Qwen. Do not run while authoring.
import torch
from knowlytix.knowledge.config import DocGMSConfig, GeometryConfig
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.knowledge.llm_backend import LocalTransformersBackend
from knowlytix.knowledge.rag.query_triples import (
    QueryTripleExtractor, schema_from_store, ASKED,
)

STORE = os.path.join(
    (os.path.join(os.path.dirname(os.getcwd()), "code") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()),
    "data", "gms_annual_report_store")
dev = "cuda" if torch.cuda.is_available() else "cpu"
store = GMSExpertStore(DocGMSConfig(store_path=STORE, geometry=GeometryConfig(d_v=64, d_u=64, m=32, d=32)), device=torch.device(dev))
assert store.load(), "trained store not found — build it with scripts/build_store.py"

vocab = schema_from_store(store)
qwen = LocalTransformersBackend("Qwen/Qwen3-4B-Instruct-2507", device=dev)
extractor_qwen = QueryTripleExtractor(qwen, vocab=vocab)

q = "How many people work in Logistics?"
qts = extractor_qwen.extract(q)
print("Qwen ->", [t.as_tuple() for t in qts])
assert any(t.tail == ASKED for t in qts)          # an asked slot was produced
# Accept has_headcount (grounded) or headcount (few-shot fallback): both name the right attribute.
assert any("headcount" in t.relation for t in qts), \
    f"grounding should bind to has_headcount; got: {[t.as_tuple() for t in qts]}"

## Exercise solution — a question Qwen mis-extracts, fixed by grounding
`What is Northwind's topline?` invites the wrong relation: `topline` is a
declared alias for revenue, but an ungrounded model emits `topline` as the
relation, which does not bind. Grounding (and binding, Ch. 7) maps it to
`has_revenue`. We show the ungrounded miss and the grounded fix.

In [ ]:
UNGROUNDED_TL = {"What is Northwind's topline?":
                 '[{"head":"northwind","relation":"topline","tail":"?"}]'}
GROUNDED_TL = {"What is Northwind's topline?":
               '[{"head":"total","relation":"has_revenue","tail":"?"}]'}

miss = QueryTripleExtractor(ScriptedBackend(UNGROUNDED_TL)).extract(
    "What is Northwind's topline?")[0]
fix = QueryTripleExtractor(ScriptedBackend(GROUNDED_TL)).extract(
    "What is Northwind's topline?")[0]
print("ungrounded:", miss.as_tuple(), "-> binds?", relation_binds(miss.relation))
print("grounded  :", fix.as_tuple(), "-> binds?", relation_binds(fix.relation))
assert not relation_binds(miss.relation)
assert relation_binds(fix.relation) and fix.tail == ASKED

## Self-check
The chapter's claim: NL questions become query triples with exactly one `?`
asked slot, and schema grounding produces a relation that binds. The final
assert proves both for a known question.

In [ ]:
qts = grounded_extractor.extract("How many people work in Logistics?")
assert len(qts) == 1
t = qts[0]
assert t.tail == ASKED                     # the asked slot
assert t.head == "logistics"               # the subject was extracted
assert t.relation == "has_headcount"       # grounded to a real graph relation
assert t.relation in expected_vocab["relations"]   # ... that binds
print("OK — query-triple extraction yields a '?' slot that binds:", t.as_tuple())